# 뉴스 데이터 전처리 및 카테고리별 임베딩

1. `title`, `summary`, `body`를 정제하고 제목·요약이 없는 행을 보완합니다. 본문은 요약 보완에만 사용합니다.
2. 제목과 요약을 각각 임베딩합니다.
3. `제목 0.7 + 요약 0.3`으로 결합 벡터를 만들고 정규화합니다.
4. 카테고리마다 제목·요약·결합 벡터 `.npy` 3개와 행 매핑용 `.json` 1개를 저장합니다. CSV는 생성하지 않습니다.

In [ ]:
%pip install -q sentence-transformers pandas tqdm

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

INPUT_DIR = Path('output')
EMBEDDING_DIR = Path('embeddings')
FILE_PATTERN = '*_enriched.csv'
MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
BATCH_SIZE = 64
MAX_SEQ_LENGTH = 512
SUMMARY_FALLBACK_CHARS = 500
TITLE_WEIGHT, SUMMARY_WEIGHT = 0.7, 0.3
TEXT_COLUMNS = ['title', 'summary', 'body']

EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def clean_text(value) -> str:
    if pd.isna(value):
        return ''
    return ' '.join(str(value).split())

def fill_empty(primary: pd.Series, fallback: pd.Series) -> pd.Series:
    return primary.mask(primary.eq(''), fallback)

def preprocess_csv(path: Path) -> tuple[pd.DataFrame, dict]:
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    missing = set(TEXT_COLUMNS) - set(frame.columns)
    if missing:
        raise ValueError(f'{path.name}: 필수 컬럼 누락 - {sorted(missing)}')

    for column in TEXT_COLUMNS:
        frame[column] = frame[column].map(clean_text)

    original_rows = len(frame)
    usable = frame[TEXT_COLUMNS].ne('').any(axis=1)
    dropped_source_rows = frame.index[~usable].tolist()
    frame = frame.loc[usable].copy()
    frame.insert(0, 'source_row', frame.index)
    frame.reset_index(drop=True, inplace=True)

    title_missing = frame['title'].eq('')
    summary_missing = frame['summary'].eq('')

    title_text = fill_empty(frame['title'], frame['summary'])
    title_text = fill_empty(title_text, frame['body'].str.slice(0, SUMMARY_FALLBACK_CHARS))

    summary_text = fill_empty(frame['summary'], frame['body'].str.slice(0, SUMMARY_FALLBACK_CHARS))
    summary_text = fill_empty(summary_text, title_text)

    frame['_embedding_title'] = title_text
    frame['_embedding_summary'] = summary_text
    frame['preprocess_note'] = [
        '; '.join(note) if note else 'original'
        for note in (
            (['title_filled'] if title else [])
            + (['summary_filled'] if summary else [])
            for title, summary in zip(title_missing, summary_missing)
        )
    ]

    report = {
        'file': path.name,
        'input_rows': original_rows,
        'usable_rows': len(frame),
        'dropped_all_empty': len(dropped_source_rows),
        'filled_title': int(title_missing.sum()),
        'filled_summary': int(summary_missing.sum()),
        'dropped_source_rows': dropped_source_rows,
    }
    return frame, report

In [ ]:
csv_files = sorted(INPUT_DIR.glob(FILE_PATTERN))
if len(csv_files) != 6:
    raise RuntimeError(f'CSV 6개를 예상했지만 {len(csv_files)}개를 찾았습니다: {[p.name for p in csv_files]}')

prepared_data = {}
preprocess_reports = []
for path in csv_files:
    category = path.stem.removesuffix('_enriched')
    prepared_data[category], report = preprocess_csv(path)
    preprocess_reports.append(report)

preprocess_report_df = pd.DataFrame(preprocess_reports)
display(preprocess_report_df.drop(columns='dropped_source_rows'))
for report in preprocess_reports:
    if report['dropped_source_rows']:
        print(f"{report['file']} 제외 원본 행: {report['dropped_source_rows']}")

In [ ]:
def normalize_rows(vectors: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / np.clip(norms, 1e-12, None)

def encode_field(model: SentenceTransformer, texts: pd.Series) -> np.ndarray:
    return model.encode(
        texts.tolist(),
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32, copy=False)

def embed_category(category: str, frame: pd.DataFrame, model: SentenceTransformer) -> dict:
    print(f'[{category}] 제목 임베딩')
    title_vectors = encode_field(model, frame['_embedding_title'])
    print(f'[{category}] 요약 임베딩')
    summary_vectors = encode_field(model, frame['_embedding_summary'])

    combined_vectors = (
        TITLE_WEIGHT * title_vectors
        + SUMMARY_WEIGHT * summary_vectors
    )
    combined_vectors = normalize_rows(combined_vectors).astype(np.float32, copy=False)

    title_path = EMBEDDING_DIR / f'{category}_title_embeddings.npy'
    summary_path = EMBEDDING_DIR / f'{category}_summary_embeddings.npy'
    combined_path = EMBEDDING_DIR / f'{category}_embeddings.npy'
    metadata_path = EMBEDDING_DIR / f'{category}_metadata.json'
    np.save(title_path, title_vectors, allow_pickle=False)
    np.save(summary_path, summary_vectors, allow_pickle=False)
    np.save(combined_path, combined_vectors, allow_pickle=False)

    metadata_columns = [
        column for column in (
            'source_row', 'article_id', 'title', 'legacy_category',
            'article_link', 'source_file', 'fetch_status', 'preprocess_note',
        )
        if column in frame.columns
    ]
    metadata = {
        'model': MODEL_NAME,
        'dimension': int(combined_vectors.shape[1]),
        'weights': {'title': TITLE_WEIGHT, 'summary': SUMMARY_WEIGHT},
        'files': {
            'title': title_path.name,
            'summary': summary_path.name,
            'combined': combined_path.name,
        },
        'rows': frame[metadata_columns].to_dict(orient='records'),
    }
    metadata_path.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    return {
        'category': category,
        'rows': len(combined_vectors),
        'dimension': combined_vectors.shape[1],
        'title_vectors': str(title_path),
        'summary_vectors': str(summary_path),
        'combined_vectors': str(combined_path),
        'metadata': str(metadata_path),
    }

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer(MODEL_NAME, device=device)
model.max_seq_length = MAX_SEQ_LENGTH
print(f'device={device} / model={MODEL_NAME} / max_seq_length={model.max_seq_length}')

## 카테고리별 임베딩

필요한 카테고리만 개별적으로 다시 실행할 수 있도록 셀을 분리했습니다.

In [ ]:
government_public_result = embed_category(
    'news_government_public', prepared_data['news_government_public'], model
)
government_public_result

In [ ]:
industry_trends_result = embed_category(
    'news_industry_trends', prepared_data['news_industry_trends'], model
)
industry_trends_result

In [ ]:
innodep_result = embed_category(
    'news_innodep', prepared_data['news_innodep'], model
)
innodep_result

In [ ]:
production_wages_result = embed_category(
    'news_production_wages', prepared_data['news_production_wages'], model
)
production_wages_result

In [ ]:
security_result = embed_category(
    'news_security', prepared_data['news_security'], model
)
security_result

In [ ]:
venture_finance_result = embed_category(
    'news_venture_finance', prepared_data['news_venture_finance'], model
)
venture_finance_result

In [ ]:
result_df = pd.DataFrame([
    government_public_result, industry_trends_result, innodep_result,
    production_wages_result, security_result, venture_finance_result,
])
display(result_df)
print(f"완료: {len(result_df)}개 카테고리 / 총 {result_df['rows'].sum():,}개 벡터")